# Кандидатогенерация для поиска услуг Авито

Для каждого поискового запроса решение выбирает до 50 объявлений из корпуса в 189 212 объявлений. Метрика — Recall@50.

**Результат на платформе: Recall@50 = 0,8998.**

Несколько независимых каналов собирают пул кандидатов (~600 на запрос). Ручная формула (RRF по рангам каналов и
признаки) упорядочивает пул, CatBoostRanker переупорядочивает топ-200, итоговый скор — смесь CatBoost и формулы.

| Канал / сигнал | Что даёт |
|---|---|
| BM25 по леммам в «гео-множестве» запроса и по всему корпусу | точные совпадения слов |
| e5-base и USER-bge-m3, дообученная на парах из train | поиск по смыслу |
| История train: похожие запросы → микрокатегории, виды и типы услуг, объявления; карта «локация поиска → локации объявлений» | поведение пользователей и география |
| CatBoostRanker (ансамбль из 2 моделей) на 47 признаках | переупорядочивание топ-200 |

Ноутбук выполняется сверху вниз. Тяжёлые артефакты (леммы, эмбеддинги, модели, таблицы кандидатов) кэшируются в `OUT_DIR`
и при повторном запуске загружаются с диска. Готовые модели CatBoost берутся из папки `models/` репозитория.
Дообучение bge-m3 вынесено в приложение A: оно выполняется один раз.

Зависимости перечислены в `requirements.txt`. Если они ещё не установлены, раскомментируйте строку ниже.

In [ ]:
# !pip install -r requirements.txt

## 1. Пути и окружение

Решение работает локально и в Google Colab. Внешние API не используются: модели загружаются из локальных папок в `OUT_DIR`.
Сеть нужна только для скачивания: готовых артефактов с Google Drive (ячейка ниже) или, в приложении A, исходных
open-source моделей с Hugging Face.

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/dataset'
else:
    DATA_DIR = os.environ.get('AVITO_DATA_DIR', './data')     # папка с тремя parquet-файлами

OUT_DIR = os.path.join(DATA_DIR, 'work')                      # кэш, модели, эмбеддинги, answer.csv
os.makedirs(OUT_DIR, exist_ok=True)
print('данные:', DATA_DIR, '|', sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')))

Недостающие файлы скачиваются из общей папки Google Drive (раздел «Готовые артефакты» в README): данные задания — в `DATA_DIR`,
дообученная bge-m3, e5-base, эмбеддинги и леммы — в `OUT_DIR`. Берутся только нужные решению файлы; всё, что уже есть
на диске, не скачивается повторно.

In [ ]:
DRIVE_FOLDER = 'https://drive.google.com/drive/folders/1tyGfyGSSqFuv-RrE-i1dz8RVgMja2tIc'
NEED_DATA = ['train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet']
NEED_WORK = ['bge-ft/', 'e5-base/', 'bgeft_docs.npy', 'e5_docs.npy',
             'q_val_bgeft.npy', 'q_val_e5.npy', 'q_bq_bgeft.npy', 'q_bq_e5.npy', 'q_tr6000_bgeft.npy', 'q_tr6000_e5.npy',
             'docs_lemmas.parquet', 'field_lemmas.parquet', 'train_vid_tip.npz']

def local_path(p):
    # путь внутри папки на Drive -> локальный путь
    return os.path.join(OUT_DIR, p[len('work/'):]) if p.startswith('work/') else os.path.join(DATA_DIR, p)

def is_needed(p):
    if p in NEED_DATA:
        return True
    name = p[len('work/'):] if p.startswith('work/') else None
    return name is not None and any(name == n or (n.endswith('/') and name.startswith(n)) for n in NEED_WORK)

missing = [n for n in NEED_DATA if not os.path.exists(os.path.join(DATA_DIR, n))] + \
          [n for n in NEED_WORK if not os.path.exists(os.path.join(OUT_DIR, n))]
if missing:
    import gdown
    print('скачиваем с Google Drive:', missing)
    for f in gdown.download_folder(DRIVE_FOLDER, skip_download=True, quiet=True):
        dst = local_path(f.path)
        if is_needed(f.path) and not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            gdown.download(id=f.id, output=dst, quiet=False)
print('данные:', sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.parquet')))

## 2. Загрузка данных

Координаты, цена и рейтинг хранятся в parquet как `Decimal`, приводим их к `float`. Запросы нормализуем:
нижний регистр, `ё → е`, только буквы и цифры. Фильтр поиска (`search_infm_params_text`) сохраняем отдельно.

In [ ]:
import os, re, json, time, gc
from decimal import Decimal
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pymorphy3
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

RNG = np.random.default_rng(42)

train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
# Длинные тексты объявлений train нужны только при дообучении (приложение A): удаляем, чтобы сэкономить память
train = train.drop(columns=['item_description_raw', 'item_infm_params_text'])
bq    = pd.read_parquet(f'{DATA_DIR}/benchmark_queries.parquet')
bi    = pd.read_parquet(f'{DATA_DIR}/benchmark_items.parquet').reset_index(drop=True)

def decimals_to_float(df):
    # Координаты, цена и рейтинг в parquet лежат как Decimal: приводим к float
    for c in df.columns:
        if df[c].dtype == object:
            s = df[c].dropna().head(100)
            if len(s) and s.map(lambda x: isinstance(x, Decimal)).all():
                df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')
for d in (train, bq, bi):
    decimals_to_float(d)

def norm_q(s):
    # Нормализация текста запроса: нижний регистр, ё -> е, только буквы и цифры
    s = str(s).lower().replace('ё', 'е')
    return ' '.join(re.sub(r'[^0-9a-zа-я]+', ' ', s).split())

train['q_norm'] = train['search_query'].map(norm_q)
bq['q_norm']    = bq['search_query'].map(norm_q)
for d in (train, bq):
    d['filt'] = d['search_infm_params_text'].fillna('').astype(str).str.strip()

N_DOCS = len(bi)
ITEM2IDX = pd.Series(np.arange(N_DOCS), index=bi['item_id'])
print(train.shape, bq.shape, bi.shape)

## 3. Лемматизация корпуса

Для BM25 слова приводятся к начальной форме (`pymorphy3`), иначе «монтаж домофонов» и «монтажа домофона» не совпадут.
Текст объявления: заголовок ×3 (главное поле), начало параметров и начало описания. Результат кэшируется.

In [ ]:
# Кэш по словам: каждое уникальное слово разбирается pymorphy3 один раз
morph = pymorphy3.MorphAnalyzer()
TOKEN_RE = re.compile(r'[a-zа-я0-9]+')
STOP = set('и в во на с со по для от до из к ко у о об а но или не за при под над без же ли бы то это как что '
           'мы вы я он она они все весь наш ваш свой быть'.split())
_lemma_cache = {}

def lemma(w):
    r = _lemma_cache.get(w)
    if r is None:
        # pymorphy3 разбирает только кириллицу; латиницу и числа оставляем как есть
        r = morph.parse(w)[0].normal_form if 'а' <= w[0] <= 'я' else w
        _lemma_cache[w] = r
    return r

def lemmatize(text):
    t = str(text).lower().replace('ё', 'е')
    return [lemma(w) for w in TOKEN_RE.findall(t) if len(w) > 1 and w not in STOP]

# Текст объявления для BM25: заголовок x3 (самое важное поле) + начало параметров + начало описания.
# Параметры содержат "Вид услуги", "Тип услуги", "Услуга", адрес — полезные ключевые слова.
CACHE = f'{OUT_DIR}/docs_lemmas.parquet'
if os.path.exists(CACHE):
    doc_tokens = pd.read_parquet(CACHE)['tokens'].tolist()
else:
    t0 = time.time()
    doc_tokens = []
    for title, params, desc in zip(bi['item_title_raw'], bi['item_infm_params_text'], bi['item_description_raw']):
        tl = lemmatize(title)
        doc_tokens.append(' '.join(tl * 3 + lemmatize(str(params)[:1000]) + lemmatize(str(desc)[:600])))
    pd.DataFrame({'tokens': doc_tokens}).to_parquet(CACHE)
    print(f'лемматизация корпуса: {time.time()-t0:.0f} c, словарь лемм {len(_lemma_cache)}')

## 4. BM25

BM25-веса считаются один раз для всех пар (документ, слово) и хранятся в разреженной матрице.
Скор запроса — сумма столбцов его слов, поэтому поиск легко ограничить любым подмножеством документов, например гео.

In [ ]:
# BM25-веса для всех пар (документ, слово); скор запроса — сумма столбцов его слов
cv = CountVectorizer(analyzer=str.split, min_df=2, dtype=np.float32)
X = cv.fit_transform(doc_tokens)
VOCAB = cv.vocabulary_

def bm25_matrix(X, k1=1.2, b=0.75):
    X = X.tocsr().astype(np.float32).copy()
    dl = np.asarray(X.sum(1)).ravel()
    avg = dl.mean()
    df = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.log(1 + (X.shape[0] - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(X.shape[0]), np.diff(X.indptr))
    tf = X.data
    X.data = idf[X.indices] * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / avg))
    return X.tocsc()   # CSC: быстрые срезы по словам

W_BM25 = bm25_matrix(X)

def query_terms(q):
    return sorted({VOCAB[t] for t in lemmatize(q) if t in VOCAB})

def bm25_scores(terms):
    if not terms:
        return np.zeros(N_DOCS, dtype=np.float32)
    return np.asarray(W_BM25[:, terms].sum(axis=1)).ravel()

print('BM25:', W_BM25.shape)

## 5. Локальная валидация

Правильные ответы бенчмарка скрыты, поэтому валидация строится из train:

* берём только поиски, где выбранное объявление **есть в корпусе** `benchmark_items` (~33 тыс. строк train), — тогда поиск идёт по тому же корпусу, что и на платформе;
* 3000 поисков отбираем так, чтобы **распределение длины запроса и доля фильтров** совпадали с бенчмарком;
* из данных для истории убираем все строки с теми же парами «запрос + локация», что в валидации.

В разделе 7 история дополнительно делается **пессимистичной**. Итоговая оценка взвешивает знакомые и новые запросы
в пропорции бенчмарка (37,5% / 62,5%). Так расхождение с платформой сократилось с ~5 до ~1,4 п.п. (подробнее — в README).

In [ ]:
# Берём только строки train, где выбранное объявление есть в корпусе: тогда поиск идёт по benchmark_items, как на платформе.
# "Поиск" = группа строк с одинаковыми (запрос, локация, фильтр). Отбираем группы так,
# чтобы длина запроса и доля фильтров совпадали с бенчмарком.
N_VAL = 3000
train['g'] = train['q_norm'] + '|' + train['search_location_id'].astype(str) + '|' + train['filt']
train['in_corpus'] = train['item_id'].isin(ITEM2IDX.index)

gold_all = train[train['in_corpus']].groupby('g')['item_id'].agg(lambda s: set(s))
gmeta = train[train['in_corpus']].drop_duplicates('g').set_index('g')[['q_norm', 'search_location_id', 'filt']]
gmeta['len_b'] = gmeta['q_norm'].str.split().str.len().clip(upper=6)
gmeta['has_f'] = (gmeta['filt'] != '').astype(int)

bq_len_b = bq['q_norm'].str.split().str.len().clip(upper=6)
target = pd.crosstab(bq_len_b, (bq['filt'] != '').astype(int), normalize=True)

val_groups = []
for (lb, hf), share in target.stack().items():
    pool = gmeta[(gmeta['len_b'] == lb) & (gmeta['has_f'] == hf)].index
    n = min(len(pool), int(round(share * N_VAL)))
    if n:
        val_groups += list(RNG.choice(pool, n, replace=False))
val = gmeta.loc[val_groups].reset_index()
val['gold'] = val['g'].map(gold_all)

# Из обучающей части убираем не только сами группы, но и все строки с той же парой (запрос, локация):
# в бенчмарке такая пара встречается в train лишь у 6.7% запросов.
val_qloc = set(zip(val['q_norm'], val['search_location_id']))
mask_val = pd.Series(list(zip(train['q_norm'], train['search_location_id'])), index=train.index).isin(val_qloc)
tr_part = train[~mask_val].copy()

print(f'валидация: {len(val)} запросов, обучающая часть: {len(tr_part)} строк')
print('доля запросов валидации, встречающихся в обучающей части:',
      round(val['q_norm'].isin(set(tr_part['q_norm'])).mean(), 3), '(в бенчмарке 0.375)')
print('доля фильтров:', round((val['filt'] != '').mean(), 3), '(в бенчмарке 0.37)')

## 6. История из train

Из 497 тыс. строк train строим три статистики:

* **гео-карта** — куда «уходят» выборы при поиске из локации L: `P(локация объявления | L)`. 83% выборов — в своей локации,
  а для регионов без своих объявлений карта указывает на ближайшие города;
* **похожие запросы** — по символьным n-граммам (ловят словоформы и опечатки);
* по похожим запросам — распределение **микрокатегорий** и «голоса» за конкретные объявления корпуса.

In [ ]:
DOC_LOC = bi['item_location_id'].to_numpy()
DOC_MC  = bi['item_microcat_id'].to_numpy()
MC_IDX  = {m: i for i, m in enumerate(np.unique(DOC_MC))}
DOC_MC_I = np.array([MC_IDX[m] for m in DOC_MC])
LOC2DOCS = pd.Series(np.arange(N_DOCS)).groupby(DOC_LOC).apply(np.array).to_dict()
PARAMS_LOW = bi['item_infm_params_text'].fillna('').str.lower().to_numpy()

def build_history(hist):
    H = {}
    # 1) Гео-карта: куда "уходят" выборы при поиске из локации L -> P(локация объявления | L)
    lm = hist.groupby(['search_location_id', 'item_location_id']).size().rename('n').reset_index()
    lm['p'] = lm['n'] / lm.groupby('search_location_id')['n'].transform('sum')
    lm = lm.sort_values(['search_location_id', 'p'], ascending=[True, False])
    lm['cum'] = lm.groupby('search_location_id')['p'].cumsum() - lm['p']
    lm = lm[(lm['cum'] < 0.97) & (lm.groupby('search_location_id').cumcount() < 10)]
    H['loc_map'] = {l: dict(zip(g['item_location_id'], g['p'])) for l, g in lm.groupby('search_location_id')}

    # 2) Похожие запросы: char n-граммы, чтобы ловить словоформы и опечатки
    uq = hist['q_norm'].value_counts()
    uq = uq.index.to_numpy()
    H['uq'] = uq
    H['qv'] = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), sublinear_tf=True, dtype=np.float32)
    H['Qm'] = H['qv'].fit_transform(uq).T.tocsr()
    qi = pd.Series(np.arange(len(uq)), index=uq)

    # 3) Матрица "запрос x микрокатегория" (сколько раз выбирали)
    mc = hist[hist['item_microcat_id'].isin(MC_IDX)]
    H['M_mc'] = sp.csr_matrix((np.ones(len(mc), np.float32),
                               (qi[mc['q_norm']].to_numpy(), mc['item_microcat_id'].map(MC_IDX).to_numpy())),
                              shape=(len(uq), len(MC_IDX)))
    # 4) Матрица "запрос x объявление корпуса" (только объявления из benchmark_items)
    it = hist[hist['in_corpus']]
    H['M_it'] = sp.csr_matrix((np.ones(len(it), np.float32),
                               (qi[it['q_norm']].to_numpy(), ITEM2IDX[it['item_id']].to_numpy())),
                              shape=(len(uq), N_DOCS))
    return H

def neighbor_weights(H, queries, k=20, batch=512):
    # Для каждого запроса: k ближайших запросов из истории с весами = косинус^2
    rows, cols, vals = [], [], []
    V = H['qv'].transform(queries)
    for s in range(0, V.shape[0], batch):
        S = (V[s:s+batch] @ H['Qm']).toarray()
        top = np.argpartition(-S, min(k, S.shape[1]-1), axis=1)[:, :k]
        for r in range(S.shape[0]):
            sims = S[r, top[r]]
            keep = sims > 0.3
            rows += [s + r] * int(keep.sum()); cols += list(top[r][keep]); vals += list(sims[keep] ** 2)
    return sp.csr_matrix((np.array(vals, np.float32), (rows, cols)), shape=(len(queries), len(H['uq'])))

def row_normalize(M):
    s = np.asarray(M.sum(1)).ravel(); s[s == 0] = 1
    return sp.diags(1 / s) @ M

## 7. Оценка качества и пессимистичная история

У ~90% объявлений бенчмарка нет истории в train. Поэтому для валидации объявления-ответы убираются из «голосов» истории,
иначе канал истории подсказывал бы ответ и завышал оценку. Здесь же определены функции подсчёта Recall@50.

In [ ]:
# Пессимистичная история: объявления-ответы валидации убраны из истории объявлений,
# потому что у большинства объявлений бенчмарка истории в train нет.
import gc

def rrf(r, k=60):
    return 1.0 / (k + r)

GOLD_IDX = [set(ITEM2IDX[list(g)]) for g in val['gold']]
N_GOLD = np.array([len(g) for g in GOLD_IDX])
seen = val['q_norm'].isin(set(tr_part['q_norm'])).to_numpy()     # знакомые запросы (37.5% в бенчмарке)

val_gold_items = set().union(*val['gold'])
tr_part.loc[tr_part['item_id'].isin(val_gold_items), 'in_corpus'] = False
H_val_pess = build_history(tr_part)
for name in ['tr_part']:          # больше не нужны: освобождаем память
    globals().pop(name, None)
gc.collect()

def mark_gold(c):
    # Для быстрой оценки помечаем правильные пары (запрос, объявление) прямо в таблице кандидатов
    gold_keys = np.array([i * N_DOCS + d for i, g in enumerate(GOLD_IDX) for d in g], dtype=np.int64)
    c['is_gold'] = np.isin(c['qi'].to_numpy(np.int64) * N_DOCS + c['doc'].to_numpy(np.int64), gold_keys)

def eval_recall(c, score, k=50):
    # Recall@50 без groupby: сортируем по (запрос, -скор) и считаем позицию внутри запроса.
    # Возвращает recall на знакомых, на новых и оценку для бенчмарка (доли 37.5% / 62.5%).
    qi = c['qi'].to_numpy()
    order = np.lexsort((-np.asarray(score), qi))
    qs, g = qi[order], c['is_gold'].to_numpy()[order]
    start = np.r_[0, np.flatnonzero(np.diff(qs)) + 1]
    pos = np.arange(len(qs)) - np.repeat(start, np.diff(np.r_[start, len(qs)]))
    r = np.bincount(qs[(pos < k) & g], minlength=len(N_GOLD)) / N_GOLD
    return r[seen].mean(), r[~seen].mean(), 0.375 * r[seen].mean() + 0.625 * r[~seen].mean()

def pool_recall(c):
    r = np.bincount(c.loc[c['is_gold'], 'qi'], minlength=len(N_GOLD)) / N_GOLD
    return r.mean()

print('история построена, знакомых запросов в валидации:', round(seen.mean(), 3))

## 8. Вид и тип услуги

Значения «вида» и «типа» услуги — закрытые списки Авито (27 видов, 913 типов). Берём их из фильтров поиска и ищем
в параметрах объявлений: вид найден у 94% объявлений корпуса, тип — у 73%. Для train коды считаются порциями и кэшируются.

In [ ]:
# Вид и тип услуги объявления. Их значения — закрытые списки Авито, поэтому берём их из фильтров поиска
# ("Вид услуги Ремонт и отделка", "Тип услуги Услуги парикмахера") и ищем в параметрах объявлений.
import pyarrow.parquet as pq

def parse_filter(f):
    v = re.search(r'Вид услуги (.+?)(?= Тип услуги|$)', f)
    t = re.search(r'Тип услуги (.+?)(?= Вид услуги|$)', f)
    return (v.group(1).strip().lower() if v else ''), (t.group(1).strip().lower() if t else '')

vid_vals, tip_vals = set(), set()
for f in pd.concat([train['filt'], bq['filt']]).unique():
    v, t = parse_filter(f)
    vid_vals.update([v] if v else []); tip_vals.update([t] if t else [])
VID_CODE = {v: i + 1 for i, v in enumerate(sorted(vid_vals))}    # 0 = неизвестно
TIP_CODE = {v: i + 1 for i, v in enumerate(sorted(tip_vals))}

def make_rx(key, vals):
    if not vals:
        return None
    alt = '|'.join(re.escape(v) for v in sorted(vals, key=len, reverse=True))   # длинные значения раньше коротких
    return re.compile(key + r' (' + alt + r')(?=\s|$)')
RX_VID, RX_TIP = make_rx('вид услуги', vid_vals), make_rx('тип услуги', tip_vals)

def codes_of(texts):
    vid = np.zeros(len(texts), np.int32); tip = np.zeros(len(texts), np.int32)
    for i, s in enumerate(texts):
        if RX_VID is not None and (m := RX_VID.search(s)):
            vid[i] = VID_CODE[m.group(1)]
        if RX_TIP is not None and (m := RX_TIP.search(s)):
            tip[i] = TIP_CODE[m.group(1)]
    return vid, tip

DOC_VID, DOC_TIP = codes_of(PARAMS_LOW)

# Для train параметры в памяти не держим: читаем колонку порциями и сразу превращаем в коды (с кэшем)
TRC = f'{OUT_DIR}/train_vid_tip.npz'
if os.path.exists(TRC):
    z = np.load(TRC); tr_vid, tr_tip = z['vid'], z['tip']
else:
    parts_v, parts_t = [], []
    for batch in pq.ParquetFile(f'{DATA_DIR}/train.parquet').iter_batches(batch_size=50_000, columns=['item_infm_params_text']):
        v, t = codes_of([str(s or '').lower() for s in batch.column(0).to_pylist()])
        parts_v.append(v); parts_t.append(t)
    tr_vid, tr_tip = np.concatenate(parts_v), np.concatenate(parts_t)
    np.savez(TRC, vid=tr_vid, tip=tr_tip)
train['item_vid'] = tr_vid[train.index.to_numpy()]
train['item_tip'] = tr_tip[train.index.to_numpy()]
print(f'видов услуг: {len(VID_CODE)}, типов: {len(TIP_CODE)} | вид найден у {np.mean(DOC_VID > 0):.1%} объявлений корпуса, '
      f'тип у {np.mean(DOC_TIP > 0):.1%}')

## 9. BM25 по параметрам и описанию, центры локаций

В общем индексе поля объявления смешаны, поэтому для ранжировщика BM25 строится отдельно по параметрам и по описанию.
Центр локации — медиана координат объявлений корпуса в ней; он нужен для признака расстояния.

In [ ]:
# Отдельные BM25 по параметрам и по описанию: в общем индексе поля смешаны, а их вклад разный.
FL = f'{OUT_DIR}/field_lemmas.parquet'
t0 = time.time()
if os.path.exists(FL):
    fl = pd.read_parquet(FL)
else:
    fl = pd.DataFrame({
        'par':  [' '.join(lemmatize(str(p)[:1000])) for p in bi['item_infm_params_text'].fillna('')],
        'desc': [' '.join(lemmatize(str(d)[:600])) for d in bi['item_description_raw'].fillna('')],
    })
    fl.to_parquet(FL)

def field_bm25(tokens):
    cvf = CountVectorizer(analyzer=str.split, min_df=2, dtype=np.float32)
    Xf = cvf.fit_transform(tokens)
    return bm25_matrix(Xf).tocsr(), cvf.vocabulary_

W_PAR, V_PAR = field_bm25(fl['par'])
W_DESC, V_DESC = field_bm25(fl['desc'])
del fl; gc.collect()

# Центр локации: медиана координат объявлений корпуса в этой локации
DOC_LAT = np.radians(bi['item_latitude'].to_numpy(np.float64))
DOC_LON = np.radians(bi['item_longitude'].to_numpy(np.float64))
_cent = bi.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()
CENTER = {l: (np.radians(a), np.radians(b)) for l, a, b in zip(_cent.index, _cent['item_latitude'], _cent['item_longitude'])}
print(f'поля: параметры {W_PAR.shape[1]} лемм, описание {W_DESC.shape[1]} лемм, {time.time()-t0:.0f} c')

## 10. Освобождение памяти

Дальше из train нужны только несколько колонок (для статистик истории), а тексты корпуса уже превращены в BM25.
Этот шаг позволяет уложить пайплайн в 12 ГБ оперативной памяти.

In [ ]:
# Освобождаем память: дальше из train нужны только 6 колонок (для истории),
# а тексты корпуса и матрица частот уже превращены в BM25
train = train[['q_norm', 'search_location_id', 'item_location_id',
               'item_microcat_id', 'item_id', 'in_corpus', 'item_vid', 'item_tip']].copy()
for name in ['doc_tokens', 'X', 'cv']:
    globals().pop(name, None)
gc.collect()
try:
    import psutil; print(f"ОЗУ: {psutil.virtual_memory().used/1e9:.1f} ГБ")
except ImportError:
    pass

## 11. Эмбеддинги

Две open-source модели, обе работают локально:

* `intfloat/multilingual-e5-base` — с префиксами `query:` / `passage:`;
* `deepvk/USER-bge-m3`, **дообученная на 200 тыс. парах «запрос → выбранное объявление»** из train (приложение A).

Исходная USER-bge-m3 тоже проверялась: рядом с дообученной её вес в формуле оказался равен 0, поэтому в финальной версии её нет.
Эмбеддинги корпуса считаются один раз (на GPU T4 ~15–25 мин на модель), эмбеддинги запросов тоже кэшируются.

In [ ]:
# Две модели эмбеддингов: e5-base и USER-bge-m3, дообученная на парах из train.
# Эмбеддинги корпуса и запросов кэшируются в OUT_DIR и при повторном запуске загружаются с диска.
import torch
from sentence_transformers import SentenceTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS = {
    'e5':    dict(path=f'{OUT_DIR}/e5-base', emb=f'{OUT_DIR}/e5_docs.npy',    qp='query: ', dp='passage: '),
    'bgeft': dict(path=f'{OUT_DIR}/bge-ft',  emb=f'{OUT_DIR}/bgeft_docs.npy', qp='',        dp=''),
}
DENSE = list(MODELS)

def item_text(title, params, desc):
    return f"{title}. {str(params)[:300]} {str(desc)[:300]}"

def query_texts(df, prefix):
    return [prefix + q + (('. ' + f) if f else '') for q, f in zip(df['q_norm'], df['filt'])]

def encode_cached(name, texts, path):
    # Эмбеддинги запросов считаем один раз и сохраняем в OUT_DIR; модель грузим, только если кэша нет
    if os.path.exists(path):
        return np.load(path)
    model = SentenceTransformer(MODELS[name]['path'], device=DEVICE)
    if DEVICE == 'cuda':
        model.half()
    model.max_seq_length = 256
    e = model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True,
                     show_progress_bar=True).astype(np.float32)
    np.save(path, e)
    del model; gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return e

D_GPU, Q_VAL, Q_BQ = {}, {}, {}
for name, m in MODELS.items():
    D = torch.from_numpy(np.load(m['emb'])).to(DEVICE)
    D_GPU[name] = D if DEVICE == 'cuda' else D.float()
    print(name, ': запросы валидации и бенчмарка')
    Q_VAL[name] = encode_cached(name, query_texts(val, m['qp']), f'{OUT_DIR}/q_val_{name}.npy')
    Q_BQ[name] = encode_cached(name, query_texts(bq, m['qp']), f'{OUT_DIR}/q_bq_{name}.npy')
    print(name, 'готово:', tuple(D_GPU[name].shape))

def dense_scores(name, q_vec):
    D = D_GPU[name]
    return (D @ torch.from_numpy(q_vec).to(DEVICE).to(D.dtype)).float().cpu().numpy()

## 12. Каналы кандидатов и ручная формула

Каналы: BM25 — топ-300 в гео-множестве и топ-100 по всему корпусу; для каждой модели эмбеддингов — топ-200 в гео-множестве
и топ-50 по всему корпусу; история похожих запросов — топ-50 объявлений.

Ручная формула: BM25 + взвешенный Reciprocal Rank Fusion по рангам каналов + нормированный косинус дообученной модели
+ гео-вероятность + совпадение фильтра. Веса подобраны покоординатным спуском на валидации.

In [ ]:
def filter_chunks(f):
    # "Тип услуги X Вид услуги Y" -> ["тип услуги x", "вид услуги y"]; пустой "Вид услуги" игнорируем
    parts = [p.strip().lower() for p in re.split(r'(?=Вид услуги|Тип услуги)', f) if p.strip()]
    return [p for p in parts if p not in ('вид услуги', 'тип услуги')]

RANK_KEYS = ['r_geo', 'r_glob', 'r_hist'] + [f'g_{m}' for m in DENSE] + [f'w_{m}' for m in DENSE]

def generate_multi(queries_df, H, Q, k_geo=300, k_glob=100, k_hist=50, k_dgeo=200, k_dglob=50):
    # Каналы: BM25 в гео-множестве и по всему корпусу, история похожих запросов,
    # и для каждой модели эмбеддингов — ближайшие по смыслу в гео-множестве (g_) и по всему корпусу (w_).
    Wn = neighbor_weights(H, queries_df['q_norm'].tolist())
    P_mc = row_normalize(Wn @ H['M_mc']).toarray()          # вероятность микрокатегории по похожим запросам
    P_it = (row_normalize(Wn) @ H['M_it']).tocsr()           # голоса истории за объявления
    out = []
    for i, (q, loc, f) in enumerate(zip(queries_df['q_norm'], queries_df['search_location_id'], queries_df['filt'])):
        s = bm25_scores(query_terms(q))
        E = {m: dense_scores(m, Q[m][i]) for m in DENSE}
        lp = dict(H['loc_map'].get(loc, {}))
        lp.setdefault(loc, max(lp.values(), default=0.5))
        geo_idx = np.concatenate([LOC2DOCS[l] for l in lp if l in LOC2DOCS] or [np.array([], int)])
        cand = {}
        def add(idx, key):
            for r, d in enumerate(idx):
                cand.setdefault(d, {})[key] = r + 1
        if len(geo_idx):
            top = geo_idx[np.argsort(-s[geo_idx])[:k_geo]]
            add(top[s[top] > 0], 'r_geo')
            for m in DENSE:
                add(geo_idx[np.argsort(-E[m][geo_idx])[:k_dgeo]], f'g_{m}')
        topg = np.argpartition(-s, k_glob)[:k_glob]
        topg = topg[np.argsort(-s[topg])]
        add(topg[s[topg] > 0], 'r_glob')
        for m in DENSE:
            tope = np.argpartition(-E[m], k_dglob)[:k_dglob]
            add(tope[np.argsort(-E[m][tope])], f'w_{m}')
        row = P_it.getrow(i)
        hist_d = dict(zip(row.indices, row.data))
        if row.nnz:
            add(row.indices[np.argsort(-row.data)][:k_hist], 'r_hist')
        chunks = filter_chunks(f)
        smax = s.max() if s.max() > 0 else 1.0
        for d, r in cand.items():
            out.append((i, d, s[d] / smax, *[E[m][d] for m in DENSE], *[r.get(k, 999) for k in RANK_KEYS],
                        hist_d.get(d, 0.0), P_mc[i, DOC_MC_I[d]], lp.get(DOC_LOC[d], 0.0),
                        np.mean([c in PARAMS_LOW[d] for c in chunks]) if chunks else -1.0))
        if i % 500 == 0:
            print(i, end=' ')
    cols = ['qi', 'doc', 'bm25'] + [f'cos_{m}' for m in DENSE] + RANK_KEYS + ['hist', 'p_mc', 'p_loc', 'filt_match']
    c = pd.DataFrame(out, columns=cols)
    del out
    # Заранее считаем RRF-слагаемые и нормированные внутри запроса косинусы: так перебор весов быстрее
    c['lex_rrf'] = rrf(c['r_geo']) + rrf(c['r_glob']) + rrf(c['r_hist'])
    for m in DENSE:
        c[f'rrf_{m}'] = rrf(c[f'g_{m}']) + rrf(c[f'w_{m}'])
        g = c.groupby('qi')[f'cos_{m}']
        c[f'z_{m}'] = (c[f'cos_{m}'] - g.transform('mean')) / (g.transform('std') + 1e-6)
    c['filt_pos'] = c['filt_match'].clip(lower=0)
    return c

def score_multi(c, w):
    # Итоговый скор: BM25 + взвешенный RRF по лексическим и смысловым каналам + признаки истории, гео и фильтра
    sc = w['bm25'] * c['bm25'].to_numpy() + 30 * w['rrf'] * c['lex_rrf'].to_numpy()
    for m in DENSE:
        sc = sc + 30 * w[f'd_{m}'] * c[f'rrf_{m}'].to_numpy() + w[f'c_{m}'] * c[f'z_{m}'].to_numpy()
    return sc + w['mc'] * c['p_mc'].to_numpy() + w['loc'] * c['p_loc'].to_numpy() + w['filt'] * c['filt_pos'].to_numpy()

## 13. Признаки для ранжировщика

Для каждой пары «запрос — кандидат» считаются:

* ранги и скоры всех каналов, косинусы моделей (сырые, нормированные внутри запроса и разрыв до лучшего), число каналов;
* **совпадение с заголовком**: BM25 только по заголовку, доля слов запроса в заголовке, «все слова в заголовке»;
* BM25 по параметрам и по описанию, доля слов запроса, найденных где-либо в объявлении;
* **расстояние до центра локации поиска** (для регионов без своих объявлений — до центра самой частой локации по гео-карте);
* **вид и тип услуги**: совпадение с фильтром запроса и их распределение по похожим запросам истории;
* история, гео, рейтинг, отзывы, цена, признаки запроса и скор ручной формулы.

В конце ячейки строятся кандидаты и признаки для валидации и считается качество ручной формулы.

In [ ]:
# CatBoost получает таблицу "запрос x кандидат": все признаки каналов (ранги, скоры, косинусы),
# признаки объявления и запроса, а также скор ручной формулы. Учится поднимать выбранное объявление наверх.
# Веса ручной формулы подобраны покоординатным спуском на валидации (оценка 0.9057, платформа 0.8912).
BEST_W3 = {'bm25': 1, 'rrf': 1, 'mc': 0, 'loc': 2, 'filt': 1, 'd_e5': 0.5, 'd_bgeft': 1, 'c_e5': 0, 'c_bgeft': 0.4}
K_TRAIN = K_RERANK = 200   # CatBoost учится и переранжирует на одних и тех же позициях: топ-200 ручной формулы

ITEM_RATING  = bi['item_rating'].to_numpy(np.float32)
ITEM_REVIEWS = np.log1p(bi['item_rating_reviews_count'].fillna(0).to_numpy(np.float32))
ITEM_PRICE   = np.log1p(bi['item_price'].fillna(0).clip(lower=0).to_numpy(np.float32))
ITEM_PHONE   = bi['item_is_phone_hidden'].to_numpy(np.float32)
ITEM_MSG     = bi['item_is_message_forbidden'].to_numpy(np.float32)
LOC_SIZE     = bi['item_location_id'].value_counts()

# Признаки совпадения запроса с заголовком объявления. Основной BM25 считается по смеси заголовка, параметров
# и описания, а пользователь выбирает прежде всего по заголовку: слово запроса в заголовке важнее, чем в описании.
t0 = time.time()
title_tokens = [' '.join(lemmatize(t)) for t in bi['item_title_raw']]
tcv = CountVectorizer(analyzer=str.split, dtype=np.float32)
XT = tcv.fit_transform(title_tokens)
TVOCAB = tcv.vocabulary_
W_TITLE = bm25_matrix(XT, k1=1.2, b=0.3).tocsr()   # BM25 только по заголовку (слабая нормировка длины)
T_BIN = (XT > 0).astype(np.float32).tocsr()          # какие леммы есть в заголовке
del title_tokens, XT, tcv; gc.collect()
print(f'индекс заголовков: {W_TITLE.shape}, {time.time()-t0:.0f} c')

def add_title_features(c, queries_df):
    qi, doc = c['qi'].to_numpy(), c['doc'].to_numpy()
    t_bm25 = np.zeros(len(c), np.float32); t_common = np.zeros(len(c), np.float32); t_cov = np.zeros(len(c), np.float32)
    rows_by_q = pd.Series(np.arange(len(c))).groupby(qi).indices
    for i, q in enumerate(queries_df['q_norm']):
        rows = rows_by_q.get(i)
        lem = set(lemmatize(q))
        tq = sorted({TVOCAB[t] for t in lem if t in TVOCAB})
        if rows is None or not tq:
            continue
        d = doc[rows]
        t_bm25[rows] = np.asarray(W_TITLE[d][:, tq].sum(axis=1)).ravel()
        t_common[rows] = np.asarray(T_BIN[d][:, tq].sum(axis=1)).ravel()
        t_cov[rows] = t_common[rows] / len(lem)
    c['t_bm25'] = t_bm25
    c['t_bm25_rel'] = t_bm25 / (pd.Series(t_bm25).groupby(qi).transform('max').to_numpy() + 1e-6)
    c['t_common'], c['t_cov'] = t_common, t_cov
    c['t_all'] = (t_cov >= 0.999).astype(np.float32)   # все слова запроса есть в заголовке
    return c

def add_cat_hist(H, hist):
    # По истории: сколько раз по запросу выбирали каждый вид и тип услуги (как для микрокатегорий)
    qpos = pd.Series(np.arange(len(H['uq'])), index=H['uq'])
    for col, n, key in [('item_vid', len(VID_CODE) + 1, 'M_vid'), ('item_tip', len(TIP_CODE) + 1, 'M_tip')]:
        h = hist[(hist[col] > 0) & hist['q_norm'].isin(qpos.index)]
        H[key] = sp.csr_matrix((np.ones(len(h), np.float32), (qpos[h['q_norm']].to_numpy(), h[col].to_numpy())),
                               shape=(len(H['uq']), n))
    return H

def haversine_rad(la1, lo1, la2, lo2):
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def search_center(loc, H):
    # центр своей локации; если в ней нет объявлений корпуса — центр самой частой локации по гео-карте
    if loc in CENTER:
        return CENTER[loc]
    lp = H['loc_map'].get(loc, {})
    for l in sorted(lp, key=lp.get, reverse=True):
        if l in CENTER:
            return CENTER[l]
    return (np.nan, np.nan)

def add_v6_features(c, queries_df, H, Wn):
    qi, doc = c['qi'].to_numpy(), c['doc'].to_numpy()
    # 1) расстояние до центра локации поиска
    cen = np.array([search_center(l, H) for l in queries_df['search_location_id']], dtype=np.float64)
    km = haversine_rad(cen[qi, 0], cen[qi, 1], DOC_LAT[doc], DOC_LON[doc])
    c['dist_log'] = np.log1p(np.where(np.isnan(km), 3000.0, km)).astype(np.float32)
    # 2) BM25 по параметрам и описанию + доля слов запроса, найденных хоть где-то в объявлении
    par = np.zeros(len(c), np.float32); dsc = np.zeros(len(c), np.float32); cov = np.zeros(len(c), np.float32)
    rows_by_q = pd.Series(np.arange(len(c))).groupby(qi).indices
    for i, q in enumerate(queries_df['q_norm']):
        rows = rows_by_q.get(i)
        lem = sorted(set(lemmatize(q)))
        if rows is None or not lem:
            continue
        d = doc[rows]
        tp = [V_PAR[t] for t in lem if t in V_PAR]; td = [V_DESC[t] for t in lem if t in V_DESC]
        if tp:
            par[rows] = np.asarray(W_PAR[d][:, tp].sum(axis=1)).ravel()
        if td:
            dsc[rows] = np.asarray(W_DESC[d][:, td].sum(axis=1)).ravel()
        found = np.zeros(len(rows), bool)
        hits = np.zeros(len(rows), np.float32)
        for t in lem:
            f = np.zeros(len(rows), bool)
            if t in TVOCAB: f |= np.asarray(T_BIN[d][:, TVOCAB[t]].todense()).ravel() > 0
            if t in V_PAR:  f |= np.asarray(W_PAR[d][:, V_PAR[t]].todense()).ravel() > 0
            if t in V_DESC: f |= np.asarray(W_DESC[d][:, V_DESC[t]].todense()).ravel() > 0
            hits += f
        cov[rows] = hits / len(lem)
    rel = lambda x: x / (pd.Series(x).groupby(qi).transform('max').to_numpy() + 1e-6)
    c['par_bm25_rel'], c['desc_bm25_rel'], c['cov_any'] = rel(par), rel(dsc), cov
    # 3) вид и тип услуги: совпадение с фильтром запроса и распределение по похожим запросам
    qv = np.array([VID_CODE.get(parse_filter(f)[0], 0) for f in queries_df['filt']])
    qt = np.array([TIP_CODE.get(parse_filter(f)[1], 0) for f in queries_df['filt']])
    dv, dt = DOC_VID[doc], DOC_TIP[doc]
    c['vid_match'] = np.where(qv[qi] > 0, (dv == qv[qi]).astype(np.float32), -1).astype(np.float32)
    c['tip_match'] = np.where(qt[qi] > 0, (dt == qt[qi]).astype(np.float32), -1).astype(np.float32)
    for key, codes, name in [('M_vid', dv, 'p_vid'), ('M_tip', dt, 'p_tip')]:
        P = row_normalize(Wn @ H[key]).tocsr()
        c[name] = np.asarray(P[qi, codes]).ravel().astype(np.float32)
    return c

def rank_in_query(qi, values):
    # Ранг значения внутри своего запроса (0 = лучший)
    return pd.Series(values).groupby(qi).rank(ascending=False, method='first').to_numpy(np.float32) - 1

def add_features(c, queries_df, H):
    qi = c['qi'].to_numpy()
    doc = c['doc'].to_numpy()
    q_loc = queries_df['search_location_id'].to_numpy()[qi]
    c['ms'] = score_multi(c, BEST_W3).astype(np.float32)          # ручная формула как признак
    c['ms_rank'] = rank_in_query(qi, c['ms'].to_numpy())
    c['same_loc'] = (DOC_LOC[doc] == q_loc).astype(np.float32)
    c['n_ch'] = sum((c[k] < 999).astype(np.float32) for k in RANK_KEYS)   # в скольких каналах найден
    for m in DENSE:
        c[f'gap_{m}'] = c[f'cos_{m}'] - c.groupby('qi')[f'cos_{m}'].transform('max')
    c['rating'], c['reviews'], c['price'] = ITEM_RATING[doc], ITEM_REVIEWS[doc], ITEM_PRICE[doc]
    c['phone_hidden'], c['msg_forbidden'] = ITEM_PHONE[doc], ITEM_MSG[doc]
    # Признаки запроса: длина, фильтр, размер своей локации, насколько похож на запросы из истории
    q_len = queries_df['q_norm'].str.split().str.len().to_numpy(np.float32)
    has_f = (queries_df['filt'] != '').to_numpy(np.float32)
    loc_sz = np.log1p(queries_df['search_location_id'].map(LOC_SIZE).fillna(0).to_numpy(np.float32))
    Wn = neighbor_weights(H, queries_df['q_norm'].tolist())
    nb = np.sqrt(Wn.max(axis=1).toarray().ravel()).astype(np.float32)
    c['q_len'], c['q_has_filter'], c['q_loc_size'], c['q_nb_sim'] = q_len[qi], has_f[qi], loc_sz[qi], nb[qi]
    c = add_title_features(c, queries_df)
    c = add_v6_features(c, queries_df, H, Wn)
    # float32 вместо float64: таблица кандидатов занимает вдвое меньше памяти
    fcols = c.select_dtypes('float64').columns
    c[fcols] = c[fcols].astype(np.float32)
    return c

FEATURES = (['bm25', 'hist', 'p_mc', 'p_loc', 'filt_match', 'lex_rrf', 'ms', 'ms_rank', 'same_loc', 'n_ch',
             'rating', 'reviews', 'price', 'phone_hidden', 'msg_forbidden', 'q_len', 'q_has_filter', 'q_loc_size', 'q_nb_sim',
             't_bm25', 't_bm25_rel', 't_common', 't_cov', 't_all',
             'dist_log', 'par_bm25_rel', 'desc_bm25_rel', 'cov_any', 'vid_match', 'tip_match', 'p_vid', 'p_tip']
            + RANK_KEYS + [f'{p}_{m}' for m in DENSE for p in ('cos', 'z', 'rrf', 'gap')])

def mark_gold_list(c, gold_idx):
    # gold_idx: для каждого запроса множество номеров правильных объявлений
    keys = np.array([i * N_DOCS + d for i, g in enumerate(gold_idx) for d in g], dtype=np.int64)
    c['is_gold'] = np.isin(c['qi'].to_numpy(np.int64) * N_DOCS + c['doc'].to_numpy(np.int64), keys)

# история для валидации (пессимистичная) + статистика по видам и типам услуг
H_val_pess = add_cat_hist(H_val_pess, train[~mask_val])

# Кандидаты валидации с признаками кэшируются в OUT_DIR
CV = f'{OUT_DIR}/cand_val_v6.parquet'
t0 = time.time()
if os.path.exists(CV):
    cand_val = pd.read_parquet(CV)
else:
    cand_val = add_features(generate_multi(val, H_val_pess, Q_VAL), val, H_val_pess)
    mark_gold(cand_val)
    cand_val.to_parquet(CV)
print(f'\nвалидация: {len(cand_val)} кандидатов, {time.time()-t0:.0f} c')
print('ручная формула:', [round(float(x), 4) for x in eval_recall(cand_val, cand_val['ms'].to_numpy())])
print(f'полнота топ-{K_RERANK} ручной формулы:',
      round(float(pool_recall(cand_val[cand_val['ms_rank'] < K_RERANK])), 4))

## 14. Обучающая выборка для CatBoost без утечек

6000 поисков из train (не пересекаются с валидацией), 2 фолда. Для каждого фолда история строится **без его строк и без его
объявлений-ответов**. В обучение берётся **только топ-200 ручной формулы**, и переранжируется ровно он же:
первая версия, куда добавлялись ответы из-за пределов топа, выучила ложное правило (подробнее — в README).

In [ ]:
# Берём поиски из train, выбранное объявление которых есть в корпусе, кроме валидации (и её пар "запрос+локация").
# Делим их на 2 фолда. Для каждого фолда история строится БЕЗ его строк и БЕЗ его объявлений-ответов:
# так признаки истории для обучающих запросов такие же "честные", как на валидации и бенчмарке.
N_TRAIN_Q = 6000   # число обучающих запросов; эмбеддинги первых 3000 берутся из кэша

tr_groups = gmeta[~pd.Series(list(zip(gmeta['q_norm'], gmeta['search_location_id'])), index=gmeta.index).isin(val_qloc)]
tr_all = tr_groups
tr_groups = tr_all.sample(n=min(N_TRAIN_Q, len(tr_all)), random_state=7).reset_index()
tr_groups['gold'] = tr_groups['g'].map(gold_all)

# Эмбеддинги обучающих запросов: уже посчитанные (первые 3000 запросов и валидация) берём из кэша по тексту,
# кодируем только недостающие.
def encode_reuse(name, texts, path, old):
    if os.path.exists(path):
        return np.load(path)
    known = {}
    for old_texts, old_path in old:
        if os.path.exists(old_path):
            known.update(zip(old_texts, np.load(old_path)))
    missing = sorted({t for t in texts if t not in known})
    print(name, f': из кэша {len(set(texts)) - len(missing)}, кодируем {len(missing)}')
    if missing:
        known.update(zip(missing, encode_cached(name, missing, path + '.missing.npy')))
    e = np.stack([known[t] for t in texts]).astype(np.float32)
    np.save(path, e)
    return e

old3000 = tr_all.sample(n=min(3000, len(tr_all)), random_state=7).reset_index()
Q_TR = {name: encode_reuse(name, query_texts(tr_groups, m['qp']), f'{OUT_DIR}/q_tr{N_TRAIN_Q}_{name}.npy',
                           [(query_texts(old3000, m['qp']), f'{OUT_DIR}/q_tr3000_{name}.npy'),
                            (query_texts(val, m['qp']), f'{OUT_DIR}/q_val_{name}.npy')])
        for name, m in MODELS.items()}

# Обучающая таблица кэшируется в OUT_DIR
CBT = f'{OUT_DIR}/cb_train_v6.parquet'
if os.path.exists(CBT):
    cb_train = pd.read_parquet(CBT)
    print('обучающая выборка из кэша:', cb_train.shape)
else:
    TR_SLIM = train[['q_norm', 'search_location_id', 'item_location_id', 'item_microcat_id', 'item_id', 'in_corpus',
                     'item_vid', 'item_tip']]
    tr_qloc_all = pd.Series(list(zip(TR_SLIM['q_norm'], TR_SLIM['search_location_id'])), index=TR_SLIM.index)
    fold = np.arange(len(tr_groups)) % 2
    parts = []
    for f in (0, 1):
        fq = tr_groups[fold == f].reset_index(drop=True)
        f_qloc = set(zip(fq['q_norm'], fq['search_location_id']))
        f_gold = set().union(*fq['gold'])
        hist_df = TR_SLIM[~tr_qloc_all.isin(f_qloc)].copy()
        hist_df.loc[hist_df['item_id'].isin(f_gold), 'in_corpus'] = False
        H_f = add_cat_hist(build_history(hist_df), hist_df)
        del hist_df; gc.collect()
        c = add_features(generate_multi(fq, H_f, {m: Q_TR[m][fold == f] for m in DENSE}), fq, H_f)
        mark_gold_list(c, [set(ITEM2IDX[list(g)]) for g in fq['gold']])
        # Только топ ручной формулы. Правильные ответы ниже топа НЕ добавляем: иначе в обучении все строки
        # с рангом 200+ окажутся положительными, и модель выучит ложное правило (утечка через отбор строк)
        c = c[c['ms_rank'] < K_TRAIN]
        c['qi'] = c['qi'] + f * 10**6                              # уникальный номер запроса между фолдами
        parts.append(c[['qi', 'is_gold'] + FEATURES].astype({k: np.float32 for k in FEATURES}))
        del c, H_f; gc.collect()
        print(f'\nфолд {f}: готово')
    cb_train = pd.concat(parts, ignore_index=True)
    del parts; gc.collect()
    # Запросы, где правильного ответа нет среди кандидатов, ничему не учат: убираем
    has_pos = cb_train.groupby('qi')['is_gold'].transform('any')
    cb_train = cb_train[has_pos].sort_values('qi').reset_index(drop=True)
    print('обучающая выборка:', cb_train.shape, '| запросов:', cb_train['qi'].nunique(),
          '| доля положительных:', round(float(cb_train['is_gold'].mean()), 4))
    cb_train.to_parquet(CBT)

## 15. Ансамбль CatBoostRanker и смесь с формулой

Две модели YetiRank (300 итераций, разные seed), предсказания усредняются. Итог — сумма нормированных внутри запроса скоров
CatBoost и формулы; вес формулы подбирается на валидации.

Если файлы моделей есть в папке `models/` репозитория (или в `OUT_DIR`), они загружаются вместо обучения: это модели,
построившие отправленный ответ. Повторное обучение на GPU недетерминировано и даёт немного другой результат.

Результат на валидации: формула 0,9048 → CatBoost 0,9201 → смесь 0,9207 (платформа: 0,8998).

In [ ]:
from catboost import CatBoostRanker, Pool

train_pool = Pool(cb_train[FEATURES], label=cb_train['is_gold'].astype(int), group_id=cb_train['qi'])
# Ансамбль двух моделей с разными seed: усредняем предсказания, это снижает случайный разброс.
# USE_SAVED_RANKERS = True: сохранённые модели загружаются вместо обучения (сначала из models/, затем из OUT_DIR);
# обучение на GPU недетерминировано, а сохранённые модели воспроизводят отправленный ответ.
USE_SAVED_RANKERS = True
rankers = []
for seed in (42, 7):
    t0 = time.time()
    saved = [p for p in (f'models/catboost_v6_seed{seed}.cbm', f'{OUT_DIR}/catboost_v6_seed{seed}.cbm')
             if os.path.exists(p)]
    if USE_SAVED_RANKERS and saved:
        r = CatBoostRanker()
        r.load_model(saved[0])
        print(f'модель seed={seed}: загружена из {saved[0]}')
    else:
        r = CatBoostRanker(loss_function='YetiRank', iterations=300, learning_rate=0.1, depth=6,
                           random_seed=seed, verbose=100, thread_count=-1,
                           task_type='GPU' if DEVICE == 'cuda' else 'CPU')
        r.fit(train_pool)
        try:
            r.save_model(f'{OUT_DIR}/catboost_v6_seed{seed}.cbm')
        except Exception:
            r.save_model(f'catboost_v6_seed{seed}.cbm')   # если OUT_DIR недоступен для записи
        print(f'модель seed={seed}: {time.time()-t0:.0f} c')
    rankers.append(r)
ranker = rankers[0]

def rerank(c):
    # CatBoost переупорядочивает топ-K_RERANK ручной формулы; остальные остаются ниже в прежнем порядке
    top = (c['ms_rank'] < K_RERANK).to_numpy()
    sc = c['ms'].to_numpy().astype(np.float64) - 1e6
    X = c.loc[top, FEATURES].astype(np.float32)
    sc[top] = np.mean([r.predict(X) for r in rankers], axis=0)
    return sc

cb_score = rerank(cand_val)
print('ручная формула:', [round(float(x), 4) for x in eval_recall(cand_val, cand_val['ms'].to_numpy())])
print('CatBoost:      ', [round(float(x), 4) for x in eval_recall(cand_val, cb_score)])

# Смесь: CatBoost + ручная формула (нормированные внутри запроса), вес подбираем на валидации
def zq(c, x):
    s = pd.Series(x).groupby(c['qi'].to_numpy())
    return ((x - s.transform('mean').to_numpy()) / (s.transform('std').to_numpy() + 1e-6))
ms_z = zq(cand_val, cand_val['ms'].to_numpy())
top_mask = (cand_val['ms_rank'] < K_RERANK).to_numpy()
cb_z = np.where(top_mask, zq(cand_val, np.where(top_mask, cb_score, np.nan)), -1e6)
best_a = max([0, 0.5, 1, 2, 3, 4, 6], key=lambda a: eval_recall(cand_val, np.nan_to_num(cb_z + a * ms_z, nan=-1e6))[2])
print('лучший вес ручной формулы в смеси:', best_a, '| смесь:',
      [round(float(x), 4) for x in eval_recall(cand_val, np.nan_to_num(cb_z + best_a * ms_z, nan=-1e6))])

imp = pd.Series(ranker.get_feature_importance(train_pool), index=FEATURES).sort_values(ascending=False)
print('\nважность признаков (топ-12):\n', imp.head(12).round(2).to_string())

## 16. Ответ для бенчмарка

История строится по всему train. Файл проверяется по требованиям формата, сохраняется в `OUT_DIR/answer.csv`
и сверяется с отправленным `answer.csv` из корня репозитория.

In [ ]:
del cand_val, cb_train, train_pool; gc.collect()
H_full = add_cat_hist(build_history(train), train)
cand_bq = add_features(generate_multi(bq, H_full, Q_BQ), bq, H_full)
cb = rerank(cand_bq)
top_mask = (cand_bq['ms_rank'] < K_RERANK).to_numpy()
cb_z = np.where(top_mask, zq(cand_bq, np.where(top_mask, cb, np.nan)), -1e6)
final = np.nan_to_num(cb_z + best_a * zq(cand_bq, cand_bq['ms'].to_numpy()), nan=-1e6)

order = np.lexsort((-final, cand_bq['qi'].to_numpy()))
qs, docs = cand_bq['qi'].to_numpy()[order], cand_bq['doc'].to_numpy()[order]
start = np.r_[0, np.flatnonzero(np.diff(qs)) + 1]
pos = np.arange(len(qs)) - np.repeat(start, np.diff(np.r_[start, len(qs)]))
keep = pos < 50
pred = pd.Series(docs[keep]).groupby(qs[keep]).agg(list)

ITEM_IDS = bi['item_id'].to_numpy()
answer = pd.DataFrame({'query_id': bq['query_id'],
                       'answer': [' '.join(ITEM_IDS[pred.get(i, [])]) for i in range(len(bq))]})
ids = answer['answer'].str.split()
assert answer['query_id'].is_unique and len(answer) == len(bq)
assert ids.map(len).le(50).all() and ids.map(lambda x: len(x) == len(set(x))).all()
assert {i for row in ids for i in row} <= set(ITEM_IDS)
answer.to_csv(f'{OUT_DIR}/answer.csv', index=False)
print('сохранено answer.csv | id в строке от', ids.map(len).min(), 'до', ids.map(len).max())

# Сверка с ответом, отправленным на платформу (answer.csv в корне репозитория)
if os.path.exists('answer.csv'):
    sent = pd.read_csv('answer.csv', dtype=str, keep_default_na=False).set_index('query_id')['answer']
    n_diff = int((answer.set_index('query_id')['answer'].reindex(sent.index) != sent).sum())
    print('совпадает с отправленным answer.csv' if n_diff == 0 and len(sent) == len(answer)
          else f'отличается от отправленного answer.csv: {n_diff} строк')

## Приложение A. Дообучение USER-bge-m3 (выполняется один раз)

Контрастивное обучение (`CachedMultipleNegativesRankingLoss`, in-batch negatives): запрос притягивается к выбранному
объявлению и отталкивается от остальных объявлений батча (127 негативов на запрос). Кэшированная версия лосса считает
батч частями по 16 и помещается в 15 ГБ видеопамяти T4.

* 200 тыс. пар из обучающей части; **запросы и объявления-ответы валидации исключены**;
* 1 эпоха, lr 2e-5, fp16, ~25 мин на T4; loss 0,99 → 0,56.

Ячейку можно запускать в отдельной сессии после разделов 1, 2 и 5 (нужен GPU).
Здесь же скачиваются исходные модели с Hugging Face — единственное обращение к сети в решении.
Вместо дообучения можно скачать готовую модель `bge-ft` (раздел «Готовые артефакты» в README).

In [ ]:
RUN_FINETUNE = False   # True — скачать исходные модели и дообучить bge-m3 (один раз, нужен GPU)

if RUN_FINETUNE:
    import torch, pyarrow as pa, pyarrow.parquet as pq
    from datasets import Dataset
    from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
    from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
    from sentence_transformers.training_args import BatchSamplers

    # Исходные open-source модели сохраняем локально, дальше решение работает без сети
    for hf_name, local in [('intfloat/multilingual-e5-base', 'e5-base'), ('deepvk/USER-bge-m3', 'user-bge-m3')]:
        if not os.path.exists(f'{OUT_DIR}/{local}'):
            SentenceTransformer(hf_name).save(f'{OUT_DIR}/{local}')

    def item_text(title, params, desc):
        return f"{title}. {str(params)[:300]} {str(desc)[:300]}"

    val_gold_items = set().union(*val['gold'])
    pairs = tr_part[~tr_part['item_id'].isin(val_gold_items)][['q_norm', 'filt', 'item_title_raw', 'item_id']]
    pairs = pairs.drop_duplicates(['q_norm', 'item_id']).sample(n=min(200_000, len(pairs)), random_state=42)

    # Параметры и описания дочитываем из parquet порциями и сразу обрезаем: экономим оперативную память
    need = np.sort(pairs.index.to_numpy())
    params_txt, desc_txt, offset = {}, {}, 0
    for batch in pq.ParquetFile(f'{DATA_DIR}/train.parquet').iter_batches(
            batch_size=50_000, columns=['item_infm_params_text', 'item_description_raw']):
        lo, hi = np.searchsorted(need, [offset, offset + batch.num_rows])
        if hi > lo:
            sub = batch.take(pa.array(need[lo:hi] - offset)).to_pandas()
            for gi, p, d in zip(need[lo:hi], sub['item_infm_params_text'], sub['item_description_raw']):
                params_txt[gi], desc_txt[gi] = str(p or '')[:300], str(d or '')[:300]
        offset += batch.num_rows

    train_ds = Dataset.from_dict({
        'anchor':   [q + (('. ' + f) if f else '') for q, f in zip(pairs['q_norm'], pairs['filt'])],
        'positive': [item_text(t, params_txt[i], desc_txt[i]) for i, t in zip(pairs.index, pairs['item_title_raw'])],
    })
    model_ft = SentenceTransformer(f'{OUT_DIR}/user-bge-m3', device='cuda')
    model_ft.max_seq_length = 128
    loss = CachedMultipleNegativesRankingLoss(model_ft, mini_batch_size=16)
    args = SentenceTransformerTrainingArguments(
        output_dir='/tmp/ft_tmp', num_train_epochs=1, per_device_train_batch_size=128, learning_rate=2e-5,
        warmup_steps=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
        logging_steps=100, save_strategy='no', report_to='none', seed=42)
    SentenceTransformerTrainer(model=model_ft, args=args, train_dataset=train_ds, loss=loss).train()
    model_ft.save(f'{OUT_DIR}/bge-ft')